# 개인이력 M2 유입 강도와 M5 결합 — 개발 실험
목표: **M5가 고정 M4의 고CLV 가격·구매금액 가중 적중값@10을 높이면서, 전체 정확도는 matched M1 대비 감소폭 1% 이내인가?**

- Dunnhumby, seed 42·43·44, 고정 100 epoch, K=1 균일 음성, 이진 그래프, 64차원·2층·L2 1e-3.
- M2: 개인이력 q_N/q_V 표현, 양성 상품을 이력에서 제외한 학습. M2 자체에 q_C는 없음.
- M5: 동일 M2를 M4 가중 BPR과 하나의 optimizer에서 공동학습. 외부 재정렬 없음.
- ρ=0.025 / 0.05 / 0.10. N/V 상대비중은 이번에는 고정.
- M4 식을 먼저 선택해야 합니다. 최근 결합은 complementary, 이전 H&M 순열 실험은 original입니다. 두 식을 같은 기준으로 취급하지 않습니다.
- M1 1개 + M4 1개 + M2 3개 + M5 3개 = seed당 최대 8개. 식·입력·설정이 일치하는 완료 결과는 재사용합니다.
- 기존 학습예산 결과에서는 고정 100 epoch만 재사용합니다. 최적 epoch 선택 없음.
- 매 epoch 체크포인트에 optimizer·난수상태를 저장합니다. 런타임을 다시 연결한 뒤 같은 노트북 재실행으로 재개합니다. Colab 세션 자체의 자동 재연결은 제공하지 않습니다.
- H&M은 후속 실행용으로 선택 가능합니다. Dunnhumby와 결과·설정을 섞지 않습니다.
- 개발 결과이며 유의성·CLV 귀속·일반화 또는 최종 성공을 주장하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess, sys
from pathlib import Path
REVIEWED_SHA = 'c773b3f995e1866eae876a942825e17df445c6d9'
REPO = Path('/content/clv-history-m5-strength-' + REVIEWED_SHA[:12])
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/jung-un/clv-m2-lightgcn-runner.git',str(REPO)], check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach',REVIEWED_SHA], check=True)
assert subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip() == REVIEWED_SHA
if 'lightgcn_clv_v3' in sys.modules:
    old = Path(sys.modules['lightgcn_clv_v3'].__file__).resolve()
    assert old.parent == REPO.resolve(), '다른 실험 모듈이 남아 있습니다. 세션을 다시 시작한 뒤 실행하세요.'
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print('실행 코드:', REVIEWED_SHA)

In [ ]:
import json
import torch
import lightgcn_clv_history_m5_strength as screen

# M4 식을 명시적으로 선택하세요. 선택 전에는 학습하지 않습니다.
M4_MODE = "선택" #@param ["선택", "complementary", "original"]
DATASET = "dunnhumby" #@param ["dunnhumby", "hm"]
assert M4_MODE != '선택', 'complementary=최근 개인이력 결합의 보완 가중 / original=이전 H&M 경제구간 적합도'
SEEDS = (42, 43, 44) if DATASET == 'dunnhumby' else (43,)
RHOS = (0.025, 0.05, 0.10)
OUT_DIR = f'/content/drive/MyDrive/논문/data/results_v3_{DATASET}_history_m5_strength_{M4_MODE}_v1'
cfg = screen.configure_strength(
    m4_mode=M4_MODE, dataset=DATASET, seeds=SEEDS, rhos=RHOS, out_dir=OUT_DIR,
)
print(json.dumps(screen.preflight_summary(cfg), ensure_ascii=False, indent=2))
print('기존 결과 탐색 폴더:', cfg.reuse_dirs)
# 기존 결과를 다른 폴더로 옮겼다면 configure_strength의 reuse_dirs에 해당 폴더들을 명시합니다.
# 병렬 실행은 런타임마다 서로 다른 seed만 지정하고 OUT_DIR은 동일하게 둡니다.
assert torch.cuda.is_available(), 'GPU 런타임이 필요합니다.'

In [ ]:
# 각 arm 완료 후 전체 원본·비교·판독 CSV/JSON이 갱신됩니다.
# 같은 설정으로 재실행하면 완료 arm을 재사용하고 미완료 arm은 마지막 epoch부터 재개합니다.
absolute = screen.run_strength(cfg)

In [ ]:
import pandas as pd
print('1) 전체·저/중/고CLV 절대지표 — 생략 없음')
view = absolute.copy()
view.attrs = {}
print(view.to_string(index=False))
print('\n2) M2−M1 / M4−M1 / M5−M1 / M5−M4 / M5−M2')
print(pd.DataFrame(absolute.attrs['comparison_records']).to_string(index=False))
print('\n3) 평균·개선 seed 수 — 유의성 판정 아님')
print(pd.DataFrame(absolute.attrs['summary_records']).to_string(index=False))
print('\n4) 각 seed와 평균의 조건 충족 여부를 별도 표시')
print(pd.DataFrame(absolute.attrs['reading_records']).to_string(index=False))
print('\n저장:', json.dumps(absolute.attrs['paths'], ensure_ascii=False, indent=2))

In [ ]:
import matplotlib.pyplot as plt
summary = pd.DataFrame(absolute.attrs['summary_records'])
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric, ref, title in [
    (axes[0], screen.PRIMARY, 'm4', 'High-CLV weighted hit@10: M5 vs M4'),
    (axes[1], 'recall@10', 'm1', 'Overall Recall@10: M5 vs M1'),
]:
    part = summary[summary.model_id.str.startswith('m5_') & summary.metric.eq(metric) & summary.reference.eq(ref)].sort_values('rho')
    ax.plot(part.rho, part.relative_change_pct, marker='o')
    ax.axhline(0, color='gray', linewidth=1)
    ax.set(xlabel='M2 rho', ylabel='Change of seed means (%)', title=title)
axes[1].axhline(-1, color='red', linestyle='--', label='-1% guard (Recall@10 only)')
axes[1].legend()
fig.suptitle(f'{DATASET} / M4={M4_MODE} / seeds={SEEDS} / development only')
plt.tight_layout()
plt.show()
# 전체 정확도 보호 조건은 Recall@10 하나가 아니라 표의 6개 지표를 모두 확인합니다.

## 검증 범위
로컬 합성 데이터로 8개 arm의 학습 경로, M4 가중 손실 전달, N/V gradient, epoch 재개 일치, 캐시 식별·판정 집계를 검사했습니다. 실제 Colab/Drive 데이터의 고비용 학습은 아직 실행하지 않았습니다. 실제 결과 재사용은 Drive 파일의 식별정보가 일치할 때만 이뤄지며, 재사용 불가 시 새 학습이 필요합니다.

M4 식이 결정되지 않았다면 설정 셀에서 멈추십시오. 두 종류의 M4를 모두 실행하라는 계획이 아닙니다.